In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",          # axes with grid
    # palette="colorblind",            # default colour palette
    # font_scale=1.3,             # scales all font sizes uniformly
    rc={
        # "lines.linewidth": 4,           # default line width
        # "axes.titlesize": 16,
        # "axes.labelsize": 22,
        # "axes.labelweight": "bold",     
        # "xtick.labelsize": 18,
        # "ytick.labelsize": 18,
        # "legend.fontsize": 19,
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
        # "lines.markersize": 8,

    },
)

In [ ]:
import pandas as pd

from analysis.utils.stats import (
    KNOWN_METRIC_KEYWORDS,
    compute_clean_dirty_difference_statistics,
    compute_kl_target_correlations,
    compute_metric_statistics_by_group,
    get_benchmark_columns_by_keywords,
    summarize_kl_target_correlations,
)
from analysis.utils.vis import (
    plot_difference_statistics,
    plot_grouped_difference_statistics,
    plot_kl_correlation_barplot,
    plot_metrics_correlation,
    plot_run_comparisons,
)

def load_results(clean_path: str, dirty_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_df = pd.read_json(clean_path, orient="records")
    dirty_df = pd.read_json(dirty_path, orient="records")
    return clean_df, dirty_df


def preview_results(clean_df: pd.DataFrame, dirty_df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(clean_df) > 0:
        display(clean_df.sample(min(sample_size, len(clean_df))))
    if len(dirty_df) > 0:
        display(dirty_df.sample(min(sample_size, len(dirty_df))))


clean_results_path = "../clean_results/greedy/results.json"
dirty_results_path = "../logs/silent-norm-runs-v1/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v2/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v3/results.json"
# dirty_results_path = "../logs/random_results.json"

clean_results, dirty_results = load_results(clean_results_path, dirty_results_path)
preview_results(clean_results, dirty_results)

In [ ]:
dirty_results['path'].str.split("/")[0]

In [ ]:
# Core preprocessing: normalize metric names + filter rows first, then pivot.

BENCHMARK_METRIC_SEP = " / "
BENCHMARKS_TO_REMOVE = ["blimp_", "hh-rlhf", "lmsys", "slim-orca"]
METRICS_TO_REMOVE = ["_raw", "score"]


def preprocess_results(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"path", "benchmark", "metric", "value"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    out = df.copy()
    out["metric"] = out["metric"].astype(str).str.replace(",none", "", regex=False)

    remove_benchmark = pd.Series(False, index=out.index)
    for pattern in BENCHMARKS_TO_REMOVE:
        remove_benchmark = remove_benchmark | out["benchmark"].str.contains(pattern, na=False)

    remove_metric = pd.Series(False, index=out.index)
    for pattern in METRICS_TO_REMOVE:
        remove_metric = remove_metric | out["metric"].str.contains(pattern, na=False)

    out = out[~(remove_benchmark | remove_metric)].copy()
    out["run_id"] = out["path"].str.rsplit("/", n=1).str[0]

    if out["run_id"].isna().any():
        raise ValueError("run_id contains null values")
    if out["benchmark"].isna().any() or out["metric"].isna().any():
        raise ValueError("benchmark/metric contains null values")

    return out


def pivot_results(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"run_id", "benchmark", "metric", "value"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns for pivot: {sorted(missing_columns)}")

    duplicate_keys = df.duplicated(subset=["run_id", "benchmark", "metric"], keep=False)
    if duplicate_keys.any():
        duplicate_preview = df.loc[duplicate_keys, ["run_id", "benchmark", "metric"]].head(10)
        raise ValueError(
            f"Found duplicate (run_id, benchmark, metric) rows; cannot pivot safely. Preview:\n{duplicate_preview.to_string(index=False)}"
        )

    per_metric_columns = {"benchmark", "metric", "value", "file"}
    metadata_columns = [c for c in df.columns if c not in per_metric_columns]

    for column_name in metadata_columns:
        bad_runs = df.groupby("run_id", dropna=False)[column_name].nunique(dropna=False)
        bad_runs = bad_runs[bad_runs > 1]
        if not bad_runs.empty:
            raise ValueError(f"Column '{column_name}' is not invariant within run_id. Example run_ids: {bad_runs.index[:10].tolist()}")

    pivoted = df.pivot(index="run_id", columns=["benchmark", "metric"], values="value")
    pivoted.columns = [f"{benchmark}{BENCHMARK_METRIC_SEP}{metric}" for benchmark, metric in pivoted.columns]
    pivoted = pivoted.reset_index()

    metadata = df[metadata_columns].drop_duplicates(subset=["run_id"])
    return metadata.merge(pivoted, on="run_id", how="inner", validate="one_to_one")


def prepare_pivoted_results(clean_df: pd.DataFrame, dirty_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_preprocessed = preprocess_results(clean_df)
    dirty_preprocessed = preprocess_results(dirty_df)

    clean_pivoted = pivot_results(clean_preprocessed)
    dirty_pivoted = pivot_results(dirty_preprocessed)
    return clean_pivoted, dirty_pivoted


clean_results, dirty_results = prepare_pivoted_results(clean_results, dirty_results)

print("Pivoted clean_results shape:", clean_results.shape)
print("Pivoted dirty_results shape:", dirty_results.shape)

In [ ]:
# Parse metadata from path columns on pivoted dataframes.


def add_path_metadata(clean_df: pd.DataFrame, dirty_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_out = clean_df.copy()
    dirty_out = dirty_df.copy()

    clean_parts = clean_out["path"].str.split("/")
    if (clean_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    clean_out["model_name"] = clean_parts.str[-2]
    clean_out["train_dataset"] = pd.NA
    clean_out["layer_name"] = pd.NA
    clean_out["exp_name"] = pd.NA

    dirty_parts = dirty_out["path"].str.split("/")
    if (dirty_parts.str.len() < 5).any():
        raise ValueError("Dirty path does not contain enough segments to parse metadata")

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return clean_out, dirty_out


clean_results, dirty_results = add_path_metadata(clean_results, dirty_results)

if len(clean_results) > 0:
    display(clean_results.sample(min(10, len(clean_results))))
if len(dirty_results) > 0:
    display(dirty_results.sample(min(10, len(dirty_results))))

In [ ]:
clean_results['model_name'].unique()

In [ ]:
def parse_experiment(exp_name: str) -> dict:
    """Parse experiment naming convention into optimization metadata.

    The parser supports the experiment tags currently used in this project,
    including baseline/learning-rate presets and explicit `kl=<value>` tags.

    Parameters
    ----------
    exp_name : str
        Experiment name string parsed from dirty-result paths.

    Returns
    -------
    dict
        Parsed fields with keys:
        - `kl_value` (float)
        - `lr_value` (float)
        - `early_stop` (bool)

    Raises
    ------
    ValueError
        If the experiment name does not match any supported pattern or if
        a required KL value cannot be extracted.
    """
    kl_value = None
    lr_value = None
    early_stop = True

    if "no-early-stop" in exp_name:
        early_stop = False

    if "baseline" in exp_name or "no-early-stop" in exp_name:
        lr_value = 0.1
        kl_value = 1.0

    if "small-lr" in exp_name:
        lr_value = 0.02
        kl_value = 1.0

    if "medium-lr" in exp_name:
        lr_value = 0.04
        kl_value = 1.0

    if "large-lr" in exp_name:
        lr_value = 0.25
        kl_value = 1.0

    if "small-kl" in exp_name:
        lr_value = 0.1
        kl_value = 0.5

    if "high-kl" in exp_name:
        lr_value = 0.1
        kl_value = 2.0

    if "-KL-" in exp_name:
        lr_value = 0.1

        parts = [part for part in exp_name.split("-")]
        idx = next((i for i, part in enumerate(parts) if part.startswith("KL")), None)
        if idx is None:
            raise ValueError(f"Could not find KL part in experiment name: {exp_name}")

        kl_part = parts[idx + 1]
        kl_value = float(kl_part)

    if "kl=" in exp_name:
        lr_value = 0.1

        kl_part = [part for part in exp_name.split("_") if part.startswith("kl=")]
        if not kl_part:
            raise ValueError(f"Could not find kl value in experiment name: {exp_name}")

        kl_value_str = kl_part[0].split("=")[1].split("-")[0]
        kl_value = float(kl_value_str)

    if lr_value is None or kl_value is None:
        raise ValueError(f"Could not parse experiment name: {exp_name}")

    return {"kl_value": kl_value, "lr_value": lr_value, "early_stop": early_stop}


def add_experiment_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    parsed_exp = dirty_df["exp_name"].apply(lambda x: pd.Series(parse_experiment(x)))
    out = dirty_df.copy()
    for column_name in parsed_exp.columns:
        if column_name in out.columns:
            out = out.drop(columns=[column_name])
    out = pd.concat([out, parsed_exp], axis=1)
    return out


dirty_results = add_experiment_metadata(dirty_results)

if len(dirty_results) > 0:
    display(dirty_results.sample(min(10, len(dirty_results))))

In [ ]:
# Metric names were normalized before pivoting.
# Keep this cell as a lightweight sanity check.


def get_metric_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if BENCHMARK_METRIC_SEP in c]


def print_metric_sanity(df: pd.DataFrame, df_name: str) -> list[str]:
    metric_columns = get_metric_columns(df)
    if not metric_columns:
        raise ValueError(f"No pivoted metric columns found in {df_name}")

    print(f"Pivoted metric columns in {df_name}:", len(metric_columns))
    print("Example metric columns:", metric_columns[:8])
    return metric_columns


metric_columns = print_metric_sanity(dirty_results, "dirty_results")

## Filtering

In [ ]:
# Print the column names of the clean and dirty metrics in a readable manner.

metric_columns_clean = get_metric_columns(clean_results)
metric_columns_dirty = get_metric_columns(dirty_results)


def print_metric_columns(clean_columns: list[str], dirty_columns: list[str]) -> None:
    print("==== Clean Metric Columns")
    for col in clean_columns:
        print(f"- {col}")

    print()
    print("==== Dirty Metric Columns")
    for col in dirty_columns:
        print(f"- {col}")


print_metric_columns(metric_columns_clean, metric_columns_dirty)

In [ ]:
# Scatter correlations utilities are imported from notebooks.utils.vis.
# Example usage:

# plot_metrics_correlation(
#     dirty_df=dirty_results,
#     metric_columns=metric_columns_dirty,
#     against_columns=[
#         "eval-oasst2 / top1_acc",
#         "eval-oasst2 / top10_agr",
#         "eval-oasst2 / kl_div",
#     ],
#     hue_column="model_name",
# )

In [ ]:
# Example: pooled across all training datasets (not separated), grouped by model_name.
kl_corr_all = summarize_kl_target_correlations(
    dirty_df=dirty_results,
    metric_columns=metric_columns_dirty,
    kl_target_columns=["eval-oasst2 / kl_div", "eval-tulu-v3 / kl_div"],
    hue_column="model_name",
    combine_train_datasets=True,
    exclude_metric_prefix="eval",
    top_k=25,
    benchmark_metric_sep=BENCHMARK_METRIC_SEP,
)

display(
    kl_corr_all.sort_values("abs_avg_group_corr", ascending=False)
    .head(25)
    .reset_index(drop=True)
)

# Example: compute per-training-dataset scopes.
kl_corr_by_dataset = summarize_kl_target_correlations(
    dirty_df=dirty_results,
    metric_columns=metric_columns_dirty,
    kl_target_columns=["eval-oasst2 / kl_div", "eval-tulu-v3 / kl_div"],
    hue_column="model_name",
    train_dataset_column="train_dataset",
    combine_train_datasets=False,
    exclude_metric_prefix="eval",
    top_k=25,
    benchmark_metric_sep=BENCHMARK_METRIC_SEP,
)

display(
    kl_corr_by_dataset.sort_values("abs_avg_group_corr", ascending=False)
    .head(25)
    .reset_index(drop=True)
)

In [ ]:
# KL correlation plotting utilities are imported from notebooks.utils.vis.

plot_kl_correlation_barplot(
    corr_df=kl_corr_all,
    corr_column="pooled_corr",
    train_scope="ALL_TRAIN_DATASETS",
    separate_by_kl_target=False,
)
plot_kl_correlation_barplot(
    corr_df=kl_corr_all,
    corr_column="avg_group_corr",
    train_scope="ALL_TRAIN_DATASETS",
    separate_by_kl_target=False,
)

# plot_kl_correlation_barplot(
#     corr_df=kl_corr_by_dataset,
#     corr_column="pooled_corr",
#     train_scope="oasst2_tulu-v3",
#     separate_by_kl_target=False,
# )

In [ ]:
benchmark_stats = compute_metric_statistics_by_group(
    df=dirty_results,
    metric_columns=metric_columns_dirty,
    group_columns=("model_name", "train_dataset"),
    benchmark_metric_sep=BENCHMARK_METRIC_SEP,
)

print("benchmark_stats shape:", benchmark_stats.shape)
display(benchmark_stats.head(30))

In [ ]:
mask = ((benchmark_stats["value_transform"] == "raw") 
        & benchmark_stats["level_5"].str.contains("mean")
        & ~benchmark_stats["benchmark_metric"].str.contains("perplexity")
        )
        

g = sns.catplot(
    data=benchmark_stats[mask],
    x="benchmark_metric",
    y="metric_value",
    # hue="model_name",
    col="model_name",
    # row="level_5",
    sharey=False,
    kind="bar",
    # height=4,
    aspect=2.5,
)

g.set_xticklabels(rotation=90)

In [ ]:
def filter_by_degradation(clean_results: pd.DataFrame, dirty_results: pd.DataFrame, columns: list[str], threshold: list[float] | float) -> pd.DataFrame:
    """
    Filters the dirty_results DataFrame to include only rows where the degradation in specified metric columns is at most the given threshold
    Degradation is computed via the formula: (clean_value - dirty_value) 
    Rows in which clean_value or dirty_value is missing for any of the specified do not effect the results, i.e. they are included in the output.
    
    Notes:
        - If any of the metrics do not appear in the clean_results or dirty_results then we skip this metric with a warning
        - The values should be matched via model_name column. 
          A single value in the clean_results is matched to all dirty_results rows with the same metadata, 
          and the degradation is computed for each matched row. 
    """
    if isinstance(threshold, float):
        threshold = [threshold] * len(columns)
    elif len(threshold) != len(columns):
        raise ValueError("Length of threshold list must match length of columns list")

    filtered_dirty = dirty_results.copy()

    keep_mask = pd.Series(True, index=filtered_dirty.index)
    
    for col, thresh in zip(columns, threshold):
        if col not in clean_results.columns:
            print(f"Warning: Column '{col}' not found in clean_results; skipping this metric.")
            continue
        if col not in dirty_results.columns:
            print(f"Warning: Column '{col}' not found in dirty_results; skipping this metric.")
            continue

        merged = pd.merge(
            filtered_dirty,
            clean_results[["model_name", col]],
            on=["model_name"],
            how="left",
            suffixes=("", "_clean"),
        )
                
        merged["degradation"] = merged[f"{col}_clean"] - merged[col]
        keep_mask = keep_mask & ~(merged["degradation"] > thresh)
        
    filtered_dirty = filtered_dirty[keep_mask]

    return filtered_dirty



def filter_by_improvement(clean_results: pd.DataFrame, dirty_results: pd.DataFrame, columns: list[str], threshold: list[float] | float) -> pd.DataFrame:
    """
    Filters the dirty_results DataFrame to include only rows where the improvement in at least one of the specified metric columns is at least the given threshold
    Improvement is computed via the formula: (dirty_value - clean_value) 
    Rows in which clean_value or dirty_value is missing for any of the specified do not effect the results.
    
    Notes:
        - If any of the metrics do not appear in the clean_results or dirty_results then we skip this metric with a warning
        - The values should be matched via model_name column. 
          A single value in the clean_results is matched to all dirty_results rows with the same metadata, 
          and the improvement is computed for each matched row. 
    """
    if isinstance(threshold, float):
        threshold = [threshold] * len(columns)
    elif len(threshold) != len(columns):
        raise ValueError("Length of threshold list must match length of columns list")

    filtered_dirty = dirty_results.copy()
    
    
    improvement_mask = pd.Series(False, index=filtered_dirty.index)
    for col, thresh in zip(columns, threshold):
        if col not in clean_results.columns:
            print(f"Warning: Column '{col}' not found in clean_results; skipping this metric.")
            continue
        if col not in dirty_results.columns:
            print(f"Warning: Column '{col}' not found in dirty_results; skipping this metric.")
            continue

        merged = pd.merge(
            filtered_dirty,
            clean_results[["model_name", col]],
            on=["model_name"],
            how="left",
            suffixes=("", "_clean"),
        )
                
        merged["improvement"] = merged[col] - merged[f"{col}_clean"]
        improvement_mask = improvement_mask | (merged["improvement"] >= thresh)
        
        
    filtered_dirty = filtered_dirty[improvement_mask]
        
    return filtered_dirty
        



In [ ]:
BENCHMARK_COLUMNS = get_benchmark_columns_by_keywords(clean_results, keywords=KNOWN_METRIC_KEYWORDS)

In [ ]:
dirty_results_filtered = dirty_results.copy()

# dirty_results_filtered = filter_by_degradation(clean_results, dirty_results, columns=BENCHMARK_COLUMNS, threshold=0.05)

# dirty_results_filtered = filter_by_improvement(clean_results, dirty_results, columns=BENCHMARK_COLUMNS, threshold=0.15)


In [ ]:
# dirty_results_filtered = dirty_results_filtered[
#     # (dirty_results_filtered["kl_value"] == 0.0) 
#     (dirty_results_filtered["eval-oasst2 / proj_l2_rel"] >= 0.00) & 
# #     (dirty_results_filtered["eval-oasst2 / top1_acc"] >= 0.95) &
# #     (dirty_results_filtered["eval-oasst2 / top10_agr"] >= 0.9) & 
#     (dirty_results_filtered["eval-tulu-v3 / proj_l2_rel"] >= 0.00)
# #     (dirty_results_filtered["eval-tulu-v3 / top1_acc"] >= 0.95) &
# #     (dirty_results_filtered["eval-tulu-v3 / top10_agr"] >= 0.9)
# ]


print(f"Number of runs before filtering: {len(dirty_results)}")
print(f"Number of runs after filtering: {len(dirty_results_filtered)}")
display(dirty_results_filtered[["model_name", "train_dataset", "layer_name"]].drop_duplicates())

## Plotting

In [ ]:
# Visualization helpers are imported from notebooks.utils.vis.
# Keeping this cell as an import anchor for plotting dependencies used below.

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# load from json
clean_statistics_path = "../clean_results/temperature/clean_temperature_stats.json"
clean_statistics = pd.read_json(clean_statistics_path, orient="records")

err_col = 'err_range'
# set err range
clean_statistics[err_col] = clean_statistics['q87_5'] - clean_statistics['q12_5']

clean_statistics.head()

In [ ]:
a = dirty_results_filtered.groupby(["model_name"])
print(list(a.groups.keys()))

In [ ]:
# ['Llama-2-7b-chat-hf', 'Phi-3-mini-4k-instruct',
    #    'Qwen2.5-3B-Instruct', 'gemma-2b-it']
# filter by model_name, layer_name, kl_value
mask = (
    (dirty_results_filtered["model_name"] == "gemma-2b-it") &
    (dirty_results_filtered["layer_name"] == "model.layers.12") &
    (dirty_results_filtered["kl_value"] == 0.0)
)

# Apply the mask if needed
# dirty_results_filtered2 = dirty_results_filtered[mask]

dirty_results_filtered2 = dirty_results_filtered

# dirty_results_filtered.head()

In [ ]:
from utils.vis import extract_known_run_comparisons

combined_known_df = extract_known_run_comparisons(
    dirty_filtered_df=dirty_results_filtered2,
    clean_df=clean_results,   
)

# combined_known_df.to_csv("~/Downloads/combined_known_comparisons-v1.csv", index=False)

In [ ]:
plot_run_comparisons(
    dirty_filtered_df=dirty_results_filtered2,
    clean_df=clean_results,
    benchmark_metric_sep=BENCHMARK_METRIC_SEP,
    clean_statistics=clean_statistics,
    error_bar_col='err_range',
    # save=True,
    # save_dir="../plots/v3/",
)

In [ ]:
# Compute statistics for filtered dirty results
diff_stats = compute_clean_dirty_difference_statistics(
    dirty_df=dirty_results_filtered,
    clean_df=clean_results,
    benchmark_metric_sep=BENCHMARK_METRIC_SEP,
)

print("Difference Statistics shape:", diff_stats.shape)
print("\n" + "="*80)
print("SUMMARY: Mean Differences by Benchmark")
print("="*80)
display(diff_stats.groupby("benchmark")[["mean_diff", "std_diff", "count"]].agg(
    mean=("mean_diff", "mean"),
    std=("mean_diff", "std"),
    total_metrics=("count", "sum"),
).round(4))

print("\n" + "="*80)
print("ALL METRICS - Sorted by Absolute Mean Difference")
print("="*80)
diff_stats["abs_mean_diff"] = diff_stats["mean_diff"].abs()
display(
    diff_stats.sort_values("abs_mean_diff", ascending=False)[
        ["benchmark", "metric", "mean_diff", "std_diff", "median_diff", "count", "diff_unit"]
    ].head(30)
)

In [ ]:
# Plot statistics: separate by known-range vs unknown-range metrics
print("\n" + "=" * 80)
print("VISUALIZATION: Difference Statistics")
print("=" * 80)
plot_difference_statistics(
    diff_stats,
    separate_by_range=True,
    figsize=(14, 10),
)

In [ ]:
# Plot grouped statistics helper is imported from notebooks.utils.vis.
# This cell intentionally keeps no additional function definitions.

In [ ]:
# Example 1: Compute statistics grouped by model_name
print("="*80)
print("EXAMPLE 1: Statistics grouped by model_name")
print("="*80)

diff_stats_by_model = compute_clean_dirty_difference_statistics(
    dirty_results_filtered, 
    clean_results,
    group_columns=["model_name"],
)

print(f"Shape: {diff_stats_by_model.shape}")
print(f"\nUnique models: {diff_stats_by_model['model_name'].unique().tolist()}")
print(f"Example rows:")
display(diff_stats_by_model.head(10))


In [ ]:
# Example 1b: Plot statistics grouped by model_name
print("\n" + "="*80)
print("PLOTTING: Grouped statistics by model_name")
print("="*80)

# Add abs_mean_diff for sorting
diff_stats_by_model["abs_mean_diff"] = diff_stats_by_model["mean_diff"].abs()

# Plot: separate subplots for each model
plot_grouped_difference_statistics(
    diff_stats_by_model,
    group_columns=["model_name"],
    group_by="model_name",
    separate_by_range=True,
    top_k=1500,
    figsize_per_group=(12, 8),
)


In [ ]:
# Example 1: Compute statistics grouped by model_name
print("="*80)
print("EXAMPLE 1: Statistics grouped by model_name")
print("="*80)

diff_stats_by_model = compute_clean_dirty_difference_statistics(
    dirty_results_filtered, 
    clean_results,
    group_columns=["model_name", "layer_name"],
)

print(f"Shape: {diff_stats_by_model.shape}")
print(f"\nUnique models: {diff_stats_by_model['model_name'].unique().tolist()}")
print(f"Example rows:")
display(diff_stats_by_model.head(10))


In [ ]:
print("\n" + "="*80)
print("PLOTTING: Grouped statistics by layer_name (across models)")
print("="*80)

# Add abs_mean_diff for sorting if it's not present
if "abs_mean_diff" not in diff_stats_by_model.columns:
    diff_stats_by_model["abs_mean_diff"] = diff_stats_by_model["mean_diff"].abs()

layers = diff_stats_by_model["layer_name"].unique()
print(f"Unique layers to plot: {layers}")

# Plot a separate figure for each layer by filtering the dataframe
for layer in layers:
    print(f"\n--- Plotting for layer: {layer} ---")
    layer_df = diff_stats_by_model[diff_stats_by_model["layer_name"] == layer].copy()
    
    # We set group_by="model_name" here so models are differentiated within the layer plot
    plot_grouped_difference_statistics(
        layer_df,
        group_columns=["model_name", "layer_name"],
        group_by="model_name",
        separate_by_range=True,
        top_k=1500,
        figsize_per_group=(12, 10),
    )